# End-to-End RAG with Bedrock Managed Knowledge Bases, AgentCore Gateway & Observability

This notebook demonstrates the complete **Bedrock Managed Knowledge Base (BMKB)** experience — from document ingestion to production-ready retrieval with centralized access control and full-stack observability.

You will create a managed KB, connect it to an **AgentCore Gateway** (MCP protocol), and monitor every layer with **CloudWatch metrics**, **OTEL spans**, and **application logs**.

### What you will build

1. **Create a BMKB** with S3 data source — Bedrock manages the vector store, IAM roles, and parsing
2. **Ingest documents** and verify retrieval with the Retrieve API
3. **Create an AgentCore Gateway** — a centralized MCP endpoint that hides KB IDs from agents
4. **Connect the KB as a Gateway Target** — agents discover and call tools without knowing the KB
5. **Configure observability** — vended log delivery and OTEL trace delivery *before* any MCP calls
6. **Create a Strands Agent** that uses the Gateway KB tool to answer questions (agent never sees KB ID)
7. **Compare Direct SDK vs Gateway** — same results, different access patterns
8. **KB Observability** — CloudWatch metrics + ingestion logs
9. **Gateway Observability** — CloudWatch metrics + OTEL spans + vended logs
10. **Clean up** all resources

### Why use a Gateway?

| Without Gateway | With Gateway |
|---|---|
| Agent knows the KB ID | KB ID is hidden — only the Gateway knows it |
| Agent calls Bedrock API directly | Agent calls Gateway URL (MCP protocol) |
| No centralized access point | Single Gateway URL for all agents |
| IAM policy per KB | IAM policy on the Gateway |
| No per-request tracing | OTEL spans for every MCP call |

### Architecture

```
                                                   ┌─────────────────┐
┌──────────┐     ┌──────────────┐     ┌────────┐   │                 │
│  Agent   │────▶│  AgentCore   │────▶│  KB    │──▶│   Managed KB    │◀── S3 Bucket
│  (IAM)   │ MCP │  Gateway     │     │ Target │   │ (vector store)  │
└──────────┘     └──────────────┘     └────────┘   └─────────────────┘
                       │                                    │
                       ▼                                    ▼
              CloudWatch Metrics              CloudWatch Metrics
              OTEL Spans (aws/spans)          Application Logs
              Vended Logs                     (ingestion tracking)
```

### Prerequisites

- AWS account with Bedrock, IAM, and CloudWatch permissions
- Model access enabled for **Managed embedding (default)** (embedding) and **Claude** (generation)
- Python 3.10+ with `boto3`, `strands-agents`, `mcp-proxy-for-aws`
- S3 bucket with documents (or the notebook creates one with synthetic data)

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

Set your region, resource names, and model ARNs. The notebook generates unique suffixes to avoid naming conflicts.

In [ ]:
import boto3
import sys
import time
import os
import json
import logging
import pprint

sys.path.insert(0, "../..")

sts_client = boto3.client('sts')
s3_client = boto3.client('s3')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration ─────────────────────────────────────────────────────
knowledge_base_name = f'bmkb-gateway-{suffix}'
bucket_name = f'bedrock-bmkb-gw-{suffix}-{account_id}'
gateway_name = f'bmkb-gw-{suffix}'

# Embedding model — use None for managed default (no extra cost)
# Or specify a custom Bedrock model (additional cost applies)
embedding_model = None  # Managed default
# embedding_model = 'amazon.titan-embed-text-v2:0'  # Custom — additional cost
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

# Gateway IAM role — auto-created below, or set your own ARN here
gateway_role_arn = None  # Set to an existing role ARN to skip auto-creation
gateway_role_name = f'AmazonBedrockGatewayRole_{suffix}'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:      {region}')
print(f'Account:     {account_id}')
print(f'KB Name:     {knowledge_base_name}')
print(f'Bucket:      {bucket_name}')
print(f'Gateway:     {gateway_name}')
print(f'GW Role:     {gateway_role_arn}')

In [ ]:
print(boto3.__version__)

## Step 2 — Create S3 bucket and upload documents

We use a synthetic Octank Financial 10K dataset. In production, this would be your own document corpus.

In [ ]:
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

In [ ]:
# Upload Octank Financial 10K report to S3
file_to_upload = '../../synthetic_dataset/octank_financial_10K.pdf'
print(f'uploading {file_to_upload} to {bucket_name}')
s3_client.upload_file(file_to_upload, bucket_name, 'octank_financial_10K.pdf')
print('Done.')

## Step 3 — Create the Bedrock Managed Knowledge Base

The `ManagedKnowledgeBase` utility handles the full lifecycle: IAM role with scoped policies, KB creation with Bedrock-managed vector store, S3 data source via the managed connector, and CloudWatch log delivery for ingestion tracking.

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_id = kb.kb_id
%store kb_id

## Step 4 — Ingest documents

Start an ingestion job. Bedrock crawls the S3 bucket, parses documents with Smart Parsing, generates embeddings, and indexes them in the managed vector store.

In [ ]:
time.sleep(30)
job = kb.start_ingestion_job()

## Step 5 — Test direct SDK retrieval (before Gateway)

Verify the KB works with direct SDK calls first.

In [ ]:
response = kb.retrieve("What is Octank Financial's total revenue?", num_results=3)

print('=== Direct SDK Retrieve ===')
for i, res in enumerate(response.get('retrievalResults', []), 1):
    print(f"{i}. score={res['score']:.4f} | {res['content']['text'][:100]}...")
print(f'Total: {len(response.get("retrievalResults", []))} chunks')

## Step 6 — Create Gateway IAM Role (auto)

The Gateway needs its own IAM role with:
- Trust policy for `bedrock-agentcore.amazonaws.com`
- Permission to call `bedrock:Retrieve` on the KB

This cell creates the role automatically. Skip if you set `gateway_role_arn` above.

In [ ]:
import json

iam_client = boto3.client('iam')

if gateway_role_arn is None:
    # Trust policy — allows AgentCore to assume this role
    gw_trust_policy = {
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
            'Action': 'sts:AssumeRole',
            'Condition': {
                'StringEquals': {'aws:SourceAccount': account_id},
            },
        }]
    }

    # Permission policy — Retrieve + read KB metadata (needed for target validation)
    gw_permission_policy = {
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'BedrockKBAccess',
            'Effect': 'Allow',
            'Action': [
                'bedrock:Retrieve',
                'bedrock:GetKnowledgeBase',
                'bedrock:ListKnowledgeBases',
            ],
            'Resource': f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/*'
        }]
    }

    try:
        role_resp = iam_client.create_role(
            RoleName=gateway_role_name,
            AssumeRolePolicyDocument=json.dumps(gw_trust_policy),
            Description=f'Gateway role for BMKB: {knowledge_base_name}',
        )
        print(f'Created role: {gateway_role_name}')
    except iam_client.exceptions.EntityAlreadyExistsException:
        role_resp = iam_client.get_role(RoleName=gateway_role_name)
        print(f'Role already exists: {gateway_role_name}')

    gateway_role_arn = role_resp['Role']['Arn']

    # Create and attach permission policy
    gw_policy_name = f'AmazonBedrockGatewayRetrievePolicy_{suffix}'
    try:
        policy_resp = iam_client.create_policy(
            PolicyName=gw_policy_name,
            PolicyDocument=json.dumps(gw_permission_policy),
        )
        policy_arn = policy_resp['Policy']['Arn']
        print(f'Created policy: {gw_policy_name}')
    except iam_client.exceptions.EntityAlreadyExistsException:
        policy_arn = f'arn:aws:iam::{account_id}:policy/{gw_policy_name}'
        print(f'Policy already exists: {gw_policy_name}')

    iam_client.attach_role_policy(RoleName=gateway_role_name, PolicyArn=policy_arn)

    # Wait for IAM propagation
    print('Waiting for IAM propagation...')
    time.sleep(30)

print(f'Gateway Role ARN: {gateway_role_arn}')

## Step 7 — Create AgentCore Gateway

The Gateway provides an MCP interface. Agents connect to the Gateway URL — they never see the KB ID.

In [ ]:
# Create Gateway with IAM authentication
gw = kb.create_gateway(
    gateway_name=gateway_name,
    gateway_role_arn=gateway_role_arn,
    auth_type='AWS_IAM',
)

gateway_id = gw['gateway_id']
gateway_url = gw['gateway_url']

print(f'\nGateway ID:  {gateway_id}')
print(f'Gateway URL: {gateway_url}')
print(f'Status:      {gw["status"]}')

## Step 8 — Create KB Target on the Gateway

Connect the KB to the Gateway as a target. The Gateway will use the `bedrock-knowledge-bases` connector with `managedSearchConfiguration`.

After this, agents can call the Gateway URL to retrieve from the KB without knowing the KB ID.

In [ ]:
target = kb.create_gateway_kb_target(
    gateway_id=gateway_id,
    target_name='kb-retrieve',
    num_results=5,
    description='Retrieve from Octank Financial KB',
)

target_id = target['target_id']
print(f'\nTarget ID: {target_id}')
print(f'Status:    {target["status"]}')
print(f'\nAgents can now connect to: {gateway_url}')
print(f'KB ID ({kb.kb_id}) is hidden from agents.')

## Step 9 — Configure Gateway Observability

Before making MCP calls, configure observability so every Gateway invocation is captured.
This sets up CloudWatch Transaction Search, vended log delivery, and trace delivery.

The Gateway emits **OTEL-compliant vended spans** server-side for every MCP invocation — no client-side instrumentation needed.

| MCP Operation | Spans Emitted |
|---------------|---------------|
| Call Tool | 2 spans: `kind:SERVER` (overall) + `kind:CLIENT` (target execution) |
| List Tools | 1 `kind:SERVER` span |
| Search Tools | 1 `kind:SERVER` span |

In [ ]:
logs_client = boto3.client('logs', region_name=region)
xray_client = boto3.client('xray', region_name=region)

gw_resource_arn = f'arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}'
gw_log_group = f'/aws/vendedlogs/bedrock-agentcore/{gateway_id}'
gw_log_group_arn = f'arn:aws:logs:{region}:{account_id}:log-group:{gw_log_group}'

def _ok(fn, label):
    try:
        result = fn()
        print(f'  ✓ {label}')
        return result
    except Exception as e:
        msg = str(e).lower()
        if any(k in msg for k in ['already exists', 'conflict', 'already set']):
            print(f'  ✓ {label} (already configured)')
        else:
            print(f'  ✗ {label}: {e}')
        return None

# ── A. CloudWatch Transaction Search ──────────────────────────────────
print('A — CloudWatch Transaction Search prerequisites')
resource_policy_doc = json.dumps({
    'Version': '2012-10-17',
    'Statement': [{
        'Sid': 'TransactionSearchXRayAccess',
        'Effect': 'Allow',
        'Principal': {'Service': 'xray.amazonaws.com'},
        'Action': 'logs:PutLogEvents',
        'Resource': [
            f'arn:aws:logs:{region}:{account_id}:log-group:aws/spans:*',
            f'arn:aws:logs:{region}:{account_id}:log-group:/aws/application-signals/data:*',
        ],
        'Condition': {
            'ArnLike': {'aws:SourceArn': f'arn:aws:xray:{region}:{account_id}:*'},
            'StringEquals': {'aws:SourceAccount': account_id},
        },
    }],
})
_ok(lambda: logs_client.put_resource_policy(policyName='AgentCoreTransactionSearchAccess', policyDocument=resource_policy_doc), 'X-Ray → aws/spans resource policy')
_ok(lambda: xray_client.update_trace_segment_destination(Destination='CloudWatchLogs'), 'Trace segment destination → CloudWatchLogs')
_ok(lambda: xray_client.update_indexing_rule(Name='Default', Rule={'Probabilistic': {'DesiredSamplingPercentage': 100}}), 'Sampling rate → 100%')

# ── B. Gateway log delivery (APPLICATION_LOGS → CWL) ──────────────────
print(f'\nB — Gateway log delivery ({gateway_id})')
_ok(lambda: logs_client.create_log_group(logGroupName=gw_log_group), f'Log group: {gw_log_group}')
_ok(lambda: logs_client.put_delivery_source(name=f'{gateway_id}-logs-source', logType='APPLICATION_LOGS', resourceArn=gw_resource_arn), 'Logs delivery source')
logs_dest = _ok(lambda: logs_client.put_delivery_destination(name=f'{gateway_id}-logs-dest', deliveryDestinationType='CWL', deliveryDestinationConfiguration={'destinationResourceArn': gw_log_group_arn}), 'Logs destination → CWL')
if logs_dest:
    _ok(lambda: logs_client.create_delivery(deliverySourceName=f'{gateway_id}-logs-source', deliveryDestinationArn=logs_dest['deliveryDestination']['arn']), 'Logs delivery connected')

# ── C. Gateway trace delivery (TRACES → X-Ray → aws/spans) ────────────
print(f'\nC — Gateway trace delivery (TRACES → X-Ray → aws/spans)')
_ok(lambda: logs_client.put_delivery_source(name=f'{gateway_id}-traces-source', logType='TRACES', resourceArn=gw_resource_arn), 'Traces delivery source')
traces_dest = _ok(lambda: logs_client.put_delivery_destination(name=f'{gateway_id}-traces-dest', deliveryDestinationType='XRAY'), 'Traces destination → X-Ray')
if traces_dest:
    _ok(lambda: logs_client.create_delivery(deliverySourceName=f'{gateway_id}-traces-source', deliveryDestinationArn=traces_dest['deliveryDestination']['arn']), 'Traces delivery connected')

print(f'\nObservability configured. All subsequent Gateway calls will be captured.')
print(f'  Vended logs → {gw_log_group}')
print(f'  OTEL spans  → aws/spans')

## Step 10 — Strands Agent with Gateway KB Tool

Now we create a [Strands](https://strandsagents.com/) agent that connects to the Gateway and uses the KB as a tool.

**How it works:**

```
┌──────────────┐         ┌──────────────┐         ┌──────────┐
│ Strands Agent │──MCP──▶│  AgentCore   │────────▶│  BMKB    │
│  (Claude)     │        │  Gateway     │         │          │
└──────────────┘         └──────────────┘         └──────────┘
       │                        │
       │                        │
  Decides WHEN to         Handles auth,
  call the tool           hides KB ID
```

- `MCPClient` — establishes the SigV4-signed connection to the Gateway (like a `boto3.client` but for MCP protocol)
- `Agent` — an LLM that reasons about your question, decides to call the KB Retrieve tool, and synthesizes the answer
- The agent **never sees the KB ID** — it only knows a tool called `kb-retrieve___Retrieve` exists on the Gateway

In [ ]:
%pip install mcp-proxy-for-aws strands-agents strands-agents-tools --quiet

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

# Connect to the Gateway via MCP (SigV4 signed)
mcp_client = MCPClient(lambda: aws_iam_streamablehttp_client(
    endpoint=gateway_url,
    aws_region=region,
    aws_service='bedrock-agentcore',
))

# Create a Strands agent with the Gateway KB tool
model = BedrockModel(
    model_id=f'{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0',
    region_name=region,
)

with mcp_client:
    # List available tools from Gateway
    tools = mcp_client.list_tools_sync()
    print('=== Gateway MCP Tools ===')
    for tool in tools:
        print(f'  Tool: {tool.tool_name}')
        desc = getattr(tool.tool_spec, 'description', '') or 'No description'
        print(f'  Description: {desc[:150]}')
    print()

    # Create the agent with Gateway tools
    agent = Agent(
        model=model,
        tools=tools,
        system_prompt='You are a helpful assistant. Use the available tools to retrieve information from the knowledge base and answer questions accurately. Always cite the source documents.',
    )

    # Ask the agent a question — it will call the Gateway KB tool automatically
    print('=== Strands Agent Query ===')
    print('Q: What is Octank Financial total revenue and what are their main risk factors?\n')
    response = agent('What is Octank Financial total revenue and what are their main risk factors?')
    print(f'\n=== Agent Answer ===')
    print(response)
    print(f'\nNote: The agent used Gateway URL ({gateway_url}) — never saw KB ID {kb.kb_id}.')

In [ ]:
# Ask a follow-up question — agent retains context
with mcp_client:
    agent = Agent(
        model=model,
        tools=mcp_client.list_tools_sync(),
        system_prompt='You are a helpful assistant. Use the available tools to retrieve information and answer questions. Be concise.',
    )
    print('Q: What strategies does Octank have for growth?\n')
    response = agent('What strategies does Octank have for growth?')
    print(f'\n=== Agent Answer ===')
    print(response)

## Step 11 — Compare: Direct SDK vs Gateway retrieval

Run the same query through both paths and compare the raw chunks returned.
Both should return the same results — the Gateway is a transparent proxy.

In [ ]:
compare_query = 'What are the key risk factors for Octank Financial?'

# ── Path 1: Direct SDK ────────────────────────────────────────────────
direct_response = kb.retrieve(compare_query, num_results=5)
direct_chunks = direct_response.get('retrievalResults', [])

print('=== Path 1: Direct SDK Retrieve ===')
for i, res in enumerate(direct_chunks, 1):
    print(f'  {i}. score={res["score"]:.4f} | {res["content"]["text"][:100]}...')
print(f'  Total: {len(direct_chunks)} chunks')

# ── Path 2: Gateway (MCP) ─────────────────────────────────────────────
print()
with mcp_client:
    tools = mcp_client.list_tools_sync()
    tool_name = tools[0].tool_name
    result = mcp_client.call_tool_sync(
        'tool_call_2',
        tool_name,
        {'retrievalQuery': {'text': compare_query}},
    )
    gw_results = json.loads(result['content'][0]['text'])
    gw_chunks = gw_results.get('retrievalResults', [])

print('=== Path 2: Gateway Retrieve (MCP) ===')
for i, res in enumerate(gw_chunks, 1):
    print(f'  {i}. score={res["score"]:.4f} | {res["content"]["text"][:100]}...')
print(f'  Total: {len(gw_chunks)} chunks')

# ── Comparison ─────────────────────────────────────────────────────────
print(f'\n=== Comparison ===')
print(f'Direct SDK: {len(direct_chunks)} chunks')
print(f'Gateway:    {len(gw_chunks)} chunks')
print(f'Note: Agent using the Gateway never sees KB ID {kb.kb_id}.')

## Step 12 — KB Observability

The Knowledge Base automatically emits metrics to CloudWatch. Ingestion logs are delivered via `enable_logging=True` (set in Step 3).

| Signal | Where | Setup needed |
|--------|-------|------|
| Metrics (Invocations, Errors, Throttles) | `AWS/Bedrock/KnowledgeBases` | Auto (needs `cloudwatch:PutMetricData` in KB role) |
| Ingestion logs (per-document status) | `/aws/vendedlogs/bedrock/.../APPLICATION_LOGS/{kb_id}` | `enable_logging=True` |

> You can also view KB metrics in the Bedrock console under Knowledge Bases → your KB → Monitoring.

In [ ]:
from datetime import datetime, timedelta
import time as _time

cw = boto3.client('cloudwatch', region_name=region)
logs_client = boto3.client('logs', region_name=region)

kb_dimension = f'knowledge-base/{kb.kb_id}'
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=1)

# Skip undocumented metrics
SKIP_METRICS = {'NumberOfVectors'}

# ── KB Metrics ────────────────────────────────────────────────────────
available = cw.list_metrics(
    Namespace='AWS/Bedrock/KnowledgeBases',
    Dimensions=[{'Name': 'KnowledgeBaseId', 'Value': kb_dimension}],
)
metrics = [m for m in available['Metrics'] if m['MetricName'] not in SKIP_METRICS]

print(f'=== KB Metrics ({kb_dimension}) ===')
if not metrics:
    print('  No metrics yet. Wait a few minutes after API calls, then re-run.')
for m in metrics:
    metric_name = m['MetricName']
    dims = {d['Name']: d['Value'] for d in m['Dimensions']}
    stat = 'Average' if 'Latency' in metric_name else 'Sum'
    resp = cw.get_metric_statistics(
        Namespace='AWS/Bedrock/KnowledgeBases', MetricName=metric_name,
        Dimensions=m['Dimensions'], StartTime=start_time, EndTime=end_time,
        Period=300, Statistics=[stat],
    )
    datapoints = resp.get('Datapoints', [])
    op = dims.get('Operation', '')
    if datapoints:
        latest = sorted(datapoints, key=lambda x: x['Timestamp'])[-1]
        print(f'  {metric_name} [{op}]: {latest[stat]:.1f}')
    else:
        print(f'  {metric_name} [{op}]: no data in last hour')

# ── KB Ingestion Logs ─────────────────────────────────────────────────
print()
kb_log_group = f'/aws/vendedlogs/bedrock/knowledge-base/APPLICATION_LOGS/{kb.kb_id}'
try:
    end_ms = int(_time.time() * 1000)
    start_ms = end_ms - (6 * 60 * 60 * 1000)
    events = logs_client.filter_log_events(
        logGroupName=kb_log_group, startTime=start_ms, endTime=end_ms, limit=10,
    ).get('events', [])
    print(f'=== KB Ingestion Logs ({len(events)} events) ===')
    for e in events[:10]:
        log = json.loads(e['message'])
        event_data = log.get('event', {})
        event_type = log.get('event_type', '')
        status = event_data.get('ingestion_job_status', event_data.get('message', ''))
        doc = event_data.get('document_title', '')
        if doc:
            print(f'  [{event_type}] {doc}: {status}')
        else:
            print(f'  [{event_type}] {status}')
except logs_client.exceptions.ResourceNotFoundException:
    print(f'=== KB Ingestion Logs ===')
    print(f'  Log group not found: {kb_log_group}')
    print(f'  This is set up by enable_logging=True in the ManagedKnowledgeBase constructor.')
except Exception as e:
    print(f'  Error: {e}')

## Step 13 — Gateway Observability

The Gateway emits metrics automatically. Vended logs and OTEL spans require the delivery setup from Step 9.

| Signal | Where | Setup needed |
|--------|-------|------|
| Metrics (Invocations, Latency, Errors, Throttles) | `AWS/Bedrock-AgentCore` | Auto (no setup) |
| Vended logs (request processing) | `/aws/vendedlogs/bedrock-agentcore/.../{gw_id}` | Delivery setup (Step 9) |
| OTEL spans (per-operation traces) | `aws/spans` | Delivery setup (Step 9) |

> You can also view Gateway metrics in the AgentCore console under Observability → Gateways. It's the same data.

> **Note on Latency:** Values shown are the **average** within the 5-minute CloudWatch period. For p50/p99, use `get_metric_data` with extended statistics.

In [ ]:
# ── Gateway Metrics ───────────────────────────────────────────────────
print('=== Gateway Metrics (AWS/Bedrock-AgentCore) ===')
for metric_name in ['Invocations', 'Latency', 'SystemErrors', 'UserErrors', 'Throttles']:
    stat = 'Average' if metric_name == 'Latency' else 'Sum'
    resp = cw.get_metric_statistics(
        Namespace='AWS/Bedrock-AgentCore', MetricName=metric_name,
        Dimensions=[{'Name': 'Operation', 'Value': 'InvokeGateway'}, {'Name': 'Protocol', 'Value': 'MCP'}],
        StartTime=start_time, EndTime=end_time, Period=300, Statistics=[stat],
    )
    datapoints = resp.get('Datapoints', [])
    if datapoints:
        latest = sorted(datapoints, key=lambda x: x['Timestamp'])[-1]
        unit = ' ms' if metric_name == 'Latency' else ''
        print(f'  {metric_name}: {latest[stat]:.1f}{unit}')
    else:
        print(f'  {metric_name}: no data in last hour')

# ── OTEL Spans ────────────────────────────────────────────────────────
print()
end_ms = int(_time.time() * 1000)
start_ms = end_ms - (2 * 60 * 60 * 1000)

try:
    resp = logs_client.filter_log_events(logGroupName='aws/spans', startTime=start_ms, endTime=end_ms, limit=10)
    events = resp.get('events', [])
    print(f'=== OTEL Spans ({len(events)} events) ===')
    if not events:
        print('  No spans yet. Wait ~10 min after Gateway calls.')
    for i, event in enumerate(events, 1):
        span = json.loads(event['message'])
        name = span.get('name', '?')
        kind = span.get('kind', '')
        attrs = span.get('attributes', {})
        latency = attrs.get('latency_ms', 'N/A')
        status_code = span.get('status', {}).get('code', '?')
        http_status = attrs.get('http.response.status_code', attrs.get('http.status_code', ''))
        duration_ns = span.get('durationNano', 0)
        duration_ms = f'{duration_ns / 1_000_000:.1f}' if duration_ns else 'N/A'
        print(f'  {i}. {name} | kind={kind} | latency={latency}ms | duration={duration_ms}ms | http={http_status} | status={status_code}')
except Exception as e:
    print(f'=== OTEL Spans: {e} ===')

# ── Vended Logs ───────────────────────────────────────────────────────
print()
gw_vended = f'/aws/vendedlogs/bedrock-agentcore/{gateway_id}'
try:
    resp = logs_client.filter_log_events(logGroupName=gw_vended, startTime=start_ms, endTime=end_ms, limit=15)
    events = [e for e in resp.get('events', []) if 'Permissions are set' not in e.get('message', '')]
    print(f'=== Gateway Vended Logs ({len(events)} events) ===')
    if not events:
        print('  No events yet. Vended logs appear after delivery is configured and a Gateway call is made.')
    for i, event in enumerate(events[:10], 1):
        entry = json.loads(event['message'])
        body = entry.get('body', {})
        log_msg = body.get('log', '')
        severity = entry.get('severityText', '')
        req_body = body.get('requestBody', '')
        method = ''
        if 'method=' in str(req_body):
            for part in str(req_body).split(','):
                if 'method=' in part:
                    method = part.split('method=')[-1].strip().rstrip('}')
                    break
        method_str = f' [{method}]' if method else ''
        print(f'  {i}. [{severity}] {log_msg}{method_str}')
except Exception as e:
    print(f'=== Gateway Vended Logs: {e} ===')

### Observability Summary

Everything lands in CloudWatch. The same data is visible in:
- **AgentCore Console** → Observability → Gateways (pre-built dashboards)
- **CloudWatch Console** → Metrics / Logs (raw access, custom dashboards)
- **SDK** (`get_metric_statistics`, `filter_log_events`) — what this notebook uses

| Component | Metrics | Logs | Spans |
|-----------|---------|------|-------|
| **KB** | `AWS/Bedrock/KnowledgeBases` (auto) | Ingestion logs (`enable_logging=True`) | — |
| **Gateway** | `AWS/Bedrock-AgentCore` (auto) | Vended logs (delivery setup) | OTEL spans in `aws/spans` (delivery setup) |

## Step 14 — Cleanup

Delete in order: Gateway Target → Gateway → Data Sources → KB → IAM → S3.

> Only run this when you're done experimenting.

In [ ]:
# Uncomment to delete everything
print('===============================Deleting all resources==============================')
# # 1. Delete Gateway resources
# kb.delete_gateway(gateway_id=gateway_id, target_id=target_id)
#
# # 2. Delete KB, data sources, IAM, S3
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)
#
# # 3. Delete Gateway IAM role (if auto-created)
# try:
#     iam_client.detach_role_policy(RoleName=gateway_role_name, PolicyArn=f'arn:aws:iam::{account_id}:policy/AmazonBedrockGatewayRetrievePolicy_{suffix}')
#     iam_client.delete_policy(PolicyArn=f'arn:aws:iam::{account_id}:policy/AmazonBedrockGatewayRetrievePolicy_{suffix}')
#     iam_client.delete_role(RoleName=gateway_role_name)
#     print(f'Deleted gateway role: {gateway_role_name}')
# except Exception as e:
#     print(f'Gateway role cleanup: {e}')

In [ ]:
# Alternative: delete by KB ID only
# from utils.managed_knowledge_base import ManagedKnowledgeBase
# ManagedKnowledgeBase.delete_kb_by_id('YOUR_KB_ID', region_name='us-west-2')

## Summary

In this notebook you built a complete, production-ready RAG pipeline:

| Layer | Component | What it does |
|-------|-----------|---|
| **Knowledge Base** | Bedrock Managed KB | Fully managed vector store — no OpenSearch or Pinecone to provision |
| **Data Source** | S3 via Managed Connector | Smart Parsing, automatic chunking and embedding |
| **Access** | AgentCore Gateway (MCP) | Centralized tool server — agents never see KB IDs |
| **Auth** | IAM + SigV4 | Gateway authenticates callers, assumes a scoped role for KB access |
| **Observability** | CloudWatch | KB metrics, Gateway metrics, OTEL spans, and ingestion application logs |

### Observability at every layer

| What | Where | How |
|------|-------|-----|
| KB Invocations, Errors, Throttles | `AWS/Bedrock/KnowledgeBases` | Auto-emitted with `cloudwatch:PutMetricData` in KB role |
| Gateway Invocations, Latency, Errors | `AWS/Bedrock-AgentCore` | Auto-emitted |
| Per-request OTEL spans | `aws/spans` + vended logs | Vended log delivery (configured in Step 9) |
| Ingestion job + document status | `/aws/vendedlogs/bedrock/.../APPLICATION_LOGS/{kb_id}` | `enable_logging=True` in KB constructor |
